# Support Ticket → Category Classifier

Classify support messages into: **Billing**, **Shipping**, **Technical**, **Refund**, **Other**.

- Load 500-row CSV, train/test split (80/20)
- Baseline: keyword rules + majority fallback
- LLM: single prompt, category-only reply
- Compare accuracy and F1

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
# Categories (must match CSV labels)
CATEGORIES = ["Billing", "Shipping", "Technical", "Refund", "Other"]
CSV_PATH = "support_tickets.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.2

In [ ]:
# Load CSV and split
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["message", "category"])
# Keep only rows whose category is in CATEGORIES
df = df[df["category"].isin(CATEGORIES)]

train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["category"]
)
train_messages = train_df["message"].tolist()
train_labels = train_df["category"].tolist()
test_messages = test_df["message"].tolist()
test_labels = test_df["category"].tolist()

print(f"Train: {len(train_messages)}, Test: {len(test_messages)}")
print("Category counts (test):", test_df["category"].value_counts().to_dict())

## Baseline: keyword rules + majority fallback

In [ ]:
from collections import Counter

# Keyword rules (lowercase match)
KEYWORDS = {
    "Billing": ["charge", "charged", "invoice", "payment", "billing", "subscription", "refund" ],
    "Shipping": ["delivery", "package", "tracking", "shipped", "ship", "order", "arrived", "address"],
    "Technical": ["login", "password", "app", "error", "crash", "website", "account", "reset", "code"],
    "Refund": ["refund", "return", "money back", "cancel"],
}

def baseline_predict(message: str) -> str:
    msg_lower = message.lower()
    for cat, words in KEYWORDS.items():
        if any(w in msg_lower for w in words):
            # Refund/Billing: prefer Refund if refund/return/cancel
            if cat == "Billing" and any(w in msg_lower for w in ["refund", "return", "money back"]):
                return "Refund"
            if cat == "Refund":
                return "Refund"
            return cat
    return majority_class

# Majority class from training set
majority_class = Counter(train_labels).most_common(1)[0][0]
baseline_preds = [baseline_predict(m) for m in test_messages]
baseline_acc = accuracy_score(test_labels, baseline_preds)
baseline_f1 = f1_score(test_labels, baseline_preds, average="weighted")

print(f"Baseline accuracy: {baseline_acc:.2%}")
print(f"Baseline F1 (weighted): {baseline_f1:.4f}")
print("\nClassification report (baseline):")
print(classification_report(test_labels, baseline_preds, zero_division=0))

## LLM classifier

In [ ]:
from openai import OpenAI

# OpenRouter: OpenAI-compatible API at openrouter.ai (use OPENROUTER_API_KEY in .env)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)
MODEL = "openai/gpt-4o-mini"  # or e.g. anthropic/claude-3-haiku, google/gemini-flash-1.5
categories_str = ", ".join(CATEGORIES)
SYSTEM = f"Classify the support message into exactly one category. Reply with only the category name, nothing else. Categories: {categories_str}."

def llm_predict(message: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": message},
        ],
        max_tokens=20,
    )
    raw = (response.choices[0].message.content or "").strip()
    # Normalize: capitalize like our labels, handle extra text
    for c in CATEGORIES:
        if c.lower() in raw.lower() or raw.lower() == c.lower():
            return c
    return majority_class  # fallback if parse fails

In [ ]:
# Run LLM on test set (may take a minute)
llm_preds = [llm_predict(m) for m in test_messages]
llm_acc = accuracy_score(test_labels, llm_preds)
llm_f1 = f1_score(test_labels, llm_preds, average="weighted")

print(f"LLM accuracy: {llm_acc:.2%}")
print(f"LLM F1 (weighted): {llm_f1:.4f}")
print("\nClassification report (LLM):")
print(classification_report(test_labels, llm_preds, zero_division=0))

## Comparison

In [ ]:
print("Summary:")
print(f"  Baseline: accuracy = {baseline_acc:.2%}, F1 = {baseline_f1:.4f}")
print(f"  LLM ({MODEL}): accuracy = {llm_acc:.2%}, F1 = {llm_f1:.4f}")
print(f"  Delta: accuracy {llm_acc - baseline_acc:+.2%}, F1 {llm_f1 - baseline_f1:+.4f}")